# 03 - Network-capacity ablation

Trains 16, 32, 64 and 128 base-filter variants on identical data, augmentation and splits.
The 32-filter model is adopted as the reported architecture on the strength of this result.

## 1. Train the four capacity variants

In [ ]:
# ==============================================================================
# Network-Capacity Ablation Study (Single Cell Execution)
# Trains 16, 32, 64, and 128 filter variants sequentially with strict memory mgmt.
# ==============================================================================
import os
import gc
import json
import time
import numpy as np
import matplotlib.pyplot as plt

# --- 1. TENSORFLOW SETUP & MEMORY FETCH ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

WINDOW_SIZE, STEP_SIZE = 128, 5
BATCH_SIZE = 32

print("\n--- Accessing Data from Memory ---")
def normalize_data(data):
    max_abs = np.max(np.abs(data))
    return np.zeros_like(data) if max_abs < 1e-9 else data / max_abs

# Good target stack
Y_full = normalize_data(final_stacked_datasets['monitoring_stage1_good_repitability'].T)

# Raw bad stack (Network input)
if 'raw_bad_stack' in globals():
    X_full = normalize_data(raw_bad_stack.T)
else:
    print("Raw bad stack not found in memory. Re-stacking...")
    import inspect
    sig = inspect.signature(process_and_stack_dataset)
    call_args = {
        'seismic_data': amplitudes_results['monitoring_stage1_bad_repitability'],
        'dataset_name': 'monitoring_stage1_bad_raw',
        'source_locations_for_this_data': locations_results['monitoring_stage1_bad_repitability']
    }
    if 'save_dir_figures' in sig.parameters:
        call_args['save_dir_figures'] = 'outputs/figures'
    if 'save_dir_numpy' in sig.parameters:
        call_args['save_dir_numpy'] = 'outputs/stacked'
    raw_bad_stack = process_and_stack_dataset(**call_args)
    X_full = normalize_data(raw_bad_stack.T)

print(f"X_full (Network Input) shape: {X_full.shape}")
print(f"Y_full (Target) shape: {Y_full.shape}")
print("Transitioning to TensorFlow Pipeline.")

# --- 5. TENSORFLOW PIPELINE ---
def sliding_window_view(arr, window_shape, step_size):
    window_h, window_w = window_shape if isinstance(window_shape, tuple) else (window_shape, window_shape)
    step_y, step_x = step_size if isinstance(step_size, tuple) else (step_size, step_size)
    arr_h, arr_w = arr.shape
    out_h, out_w = (arr_h - window_h) // step_y + 1, (arr_w - window_w) // step_x + 1
    stride_y, stride_x = arr.strides
    return np.lib.stride_tricks.as_strided(
        arr, shape=(out_h, out_w, window_h, window_w), strides=(stride_y * step_y, stride_x * step_x, stride_y, stride_x)
    )

X_view = sliding_window_view(X_full, (WINDOW_SIZE, WINDOW_SIZE), STEP_SIZE)
Y_view = sliding_window_view(Y_full, (WINDOW_SIZE, WINDOW_SIZE), STEP_SIZE)
X_segments = X_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]
Y_segments = Y_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]

X_train, X_temp, Y_train, Y_temp = train_test_split(X_segments, Y_segments, test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.3, random_state=42)

random_flipper = tf.keras.layers.RandomFlip("horizontal")
random_rotator = tf.keras.layers.RandomRotation(factor=0.03)
random_translator = tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05)

def augment(x, y):
    images = tf.concat([x, y], axis=-1)
    images = random_flipper(images)
    images = random_rotator(images)
    images = random_translator(images)
    x, y = tf.split(images, num_or_size_splits=2, axis=-1)
    if tf.random.uniform(()) > 0.5:
        noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.05, dtype=x.dtype)
        x = x + noise
    return x, y

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, Y_train))
            .cache().shuffle(1000).batch(BATCH_SIZE)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((X_val, Y_val)).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE))

# --- 6. MODEL BUILDER ---
@tf.keras.utils.register_keras_serializable()
def ssim_metric(y_true, y_pred):
    # Normalized data ranges from -1 to 1, so the max dynamic range is 2.0
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=2.0))

def build_unet(base_filters):
    f = base_filters
    inputs = layers.Input(shape=(WINDOW_SIZE, WINDOW_SIZE, 1))
    def block(x, n):
        x = layers.Conv2D(n, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(n, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        return x
    c1 = block(inputs, f);   p1 = layers.MaxPooling2D()(c1)
    c2 = block(p1, f*2);     p2 = layers.MaxPooling2D()(c2)
    c3 = block(p2, f*4);     p3 = layers.MaxPooling2D()(c3)
    c4 = block(p3, f*8);     p4 = layers.MaxPooling2D()(c4)
    c5 = block(p4, f*16)
    u6 = layers.Conv2D(f*8, 3, padding='same', activation='relu')(layers.UpSampling2D()(c5))
    c6 = block(layers.Concatenate()([u6, c4]), f*8)
    u7 = layers.Conv2D(f*4, 3, padding='same', activation='relu')(layers.UpSampling2D()(c6))
    c7 = block(layers.Concatenate()([u7, c3]), f*4)
    u8 = layers.Conv2D(f*2, 3, padding='same', activation='relu')(layers.UpSampling2D()(c7))
    c8 = block(layers.Concatenate()([u8, c2]), f*2)
    u9 = layers.Conv2D(f, 3, padding='same', activation='relu')(layers.UpSampling2D()(c8))
    c9 = block(layers.Concatenate()([u9, c1]), f)
    outputs = layers.Conv2D(1, 1, activation='linear')(c9)
    model = models.Model(inputs, outputs)

    # Compile with MAE as the mathematical loss, but SSIM as the monitored metric
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipvalue=1.0),
                  loss=tf.keras.losses.MeanAbsoluteError(), metrics=[ssim_metric])
    return model

class IncrementalHistorySaver(Callback):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.history = {'loss': [], 'ssim_metric': [], 'val_loss': [], 'val_ssim_metric': []}
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for k in self.history:
            if k in logs: self.history[k].append(float(logs[k]))
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, 'w') as f: json.dump(self.history, f, indent=2)

# --- 7. TRAINING LOOP ---
trained_labels = []
histories = {}

os.makedirs('outputs/capacity_ablation/training_logs', exist_ok=True)

for base_filters in [16, 32, 64, 128]:
    label = f'filters_{base_filters}'
    print(f"\n{'='*70}\nTraining capacity variant: base_filters={base_filters}\n{'='*70}")

    model = build_unet(base_filters)
    history_path = f'outputs/capacity_ablation/training_logs/history_{label}.json'
    saver = IncrementalHistorySaver(history_path)

    # We set mode='max' because we want the HIGHEST possible SSIM score!
    early_stop = EarlyStopping(monitor='val_ssim_metric', mode='max', patience=5, restore_best_weights=True, verbose=1)
    checkpoint_path = f'outputs/capacity_ablation/model_{label}.keras'
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_ssim_metric', mode='max', save_best_only=True, verbose=0)

    # Optional safety bypass if already trained:
    if os.path.exists(checkpoint_path) and os.path.exists(history_path):
        print(f"  Model {label} already exists. Loading history from disk to save time.")
        with open(history_path, 'r') as f:
            histories[label] = json.load(f)
        trained_labels.append(label)
    else:
        t0 = time.time()
        model.fit(train_ds, validation_data=val_ds, epochs=150, callbacks=[early_stop, checkpoint, saver], verbose=1)
        print(f"  Done in {time.time()-t0:.0f}s, {len(saver.history['loss'])} epochs, {model.count_params():,} parameters.")
        trained_labels.append(label)
        histories[label] = saver.history

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# --- 8. EVALUATION & PLOTTING ---
def calculate_nrms(data1, data2):
    rms_diff = np.sqrt(np.mean((data1 - data2) ** 2))
    denom = np.sqrt(np.mean(data1 ** 2)) + np.sqrt(np.mean(data2 ** 2))
    return 0.0 if denom == 0 else 200.0 * rms_diff / denom

# Plot the Validation SSIM curves instead of MAE
plt.figure(figsize=(9, 6))
for label, h in histories.items():
    plt.plot(h['val_ssim_metric'], label=label)
plt.xlabel('Epoch')
plt.ylabel('Validation SSIM')
plt.legend()
plt.title('Validation SSIM by Network Capacity')
os.makedirs('outputs/capacity_ablation/figures', exist_ok=True)
plt.savefig('outputs/capacity_ablation/figures/capacity_val_ssim_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n{'Capacity':15s} {'Params':>14s} {'Epochs':>8s} {'Test NRMS':>11s} {'Test SSIM':>11s}")
results = {}
for label in trained_labels:
    model = tf.keras.models.load_model(f'outputs/capacity_ablation/model_{label}.keras', compile=False)
    pred = model.predict(X_test, verbose=0)
    pred_flat = pred.reshape(pred.shape[0], -1)
    true_flat = Y_test.reshape(Y_test.shape[0], -1)

    nrms = calculate_nrms(pred_flat, true_flat)
    s = np.mean([ssim(Y_test[i,...,0], pred[i,...,0], data_range=Y_test[i,...,0].max() - Y_test[i,...,0].min() + 1e-9)
                 for i in range(min(len(Y_test), 200))])

    results[label] = {'params': model.count_params(), 'epochs': len(histories[label]['loss']), 'nrms': nrms, 'ssim': s}
    print(f"{label:15s} {model.count_params():>14,} {len(histories[label]['loss']):>8d} {nrms:>10.2f}% {s:>11.3f}")

    del model
    tf.keras.backend.clear_session()
    gc.collect()

with open('outputs/capacity_ablation/capacity_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nProcess Complete! Ablation results saved to outputs/capacity_ablation/capacity_results.json.")

## 2. Validation curves (reads saved histories, no retraining needed)

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

histories = {}
for p in sorted(glob.glob('outputs/capacity_ablation/training_logs/history_filters_*.json'),
                key=lambda s: int(s.split('_')[-1].split('.')[0])):
    n = int(p.split('_')[-1].split('.')[0])
    with open(p) as f:
        histories[n] = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
colors = {16: '#1b9e77', 64: '#d95f02', 32: '#7570b3', 128: '#666666'}

for n, h in histories.items():
    style = dict(color=colors.get(n, None), lw=2.2 if n == 64 else 1.5)
    axes[0].plot(range(1, len(h['val_loss']) + 1), h['val_loss'],
                 label=f'{n} filters', **style)
    axes[1].plot(range(1, len(h['val_ssim_metric']) + 1), h['val_ssim_metric'],
                 label=f'{n} filters', **style)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Validation loss (MAE)', fontsize=12)
axes[0].set_yscale('log')
axes[0].set_title('(a) Validation loss', fontsize=13)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation SSIM', fontsize=12)
axes[1].set_title('(b) Validation SSIM', fontsize=13)

for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)

fig.tight_layout()
os.makedirs('outputs/capacity_ablation/figures', exist_ok=True)
fig.savefig('outputs/capacity_ablation/figures/capacity_curves.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()